In [205]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.model_selection import train_test_split, cross_val_predict, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_iris

In [206]:
df_preview = load_iris()
df_preview.keys()

dict_keys(['data', 'target', 'frame', 'target_names', 'DESCR', 'feature_names', 'filename', 'data_module'])

In [207]:
df = load_iris(as_frame=True).frame

In [208]:
df['species'] = df['target'].map(dict(enumerate(df_preview.target_names)))
df = df.drop(columns='target')

## 2D Graph

In [209]:
px.scatter_matrix(df, color='target', height=800)

ValueError: Value of 'color' is not the name of a column in 'data_frame'. Expected one of ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)', 'species'] but received: target

## 3D graph

In [210]:
threed = px.scatter_3d(
    df,
    x='sepal length (cm)',
    y='sepal width (cm)',
    z='petal width (cm)',
    color='species',
)
threed.update_layout(height=500, width=1100)
threed.update_traces(
    marker=dict(
        size=8,
        line=dict(color='black', width=1)
    ),
)
threed.show()

In [211]:
y = df.pop('species')
x = df.copy()

In [212]:
xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.2, random_state=42)

In [213]:
param_grid = {
    'model__n_neighbors' : [2, 3, 4, 5],
    'model__weights': ['uniform', 'distance'],
    'model__metric' : ['euclidean', 'manhattan']
}

In [214]:
classification_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', KNeighborsClassifier())
])

In [215]:
grid = GridSearchCV(classification_pipeline, param_grid, cv=5, scoring="accuracy")
grid.fit(xtrain, ytrain)
grid.best_params_

{'model__metric': 'euclidean',
 'model__n_neighbors': 3,
 'model__weights': 'uniform'}

In [216]:
best_model = grid.best_estimator_
best_model.fit(xtrain, ytrain)
ypred = cross_val_predict(best_model, x, y, cv=5)


In [217]:
print(f"Classification report : \n {classification_report(ytest, ypred)}")

ValueError: Found input variables with inconsistent numbers of samples: [30, 150]